# PD03 · Integrar EVA con DIVIPOLA

[Índice y distribución de las cuatro horas](../ejercicios/README.md) · [Guía de pandas](../docs/manual-pandas.md)

**Tiempo de trabajo: 70 minutos.** El objetivo es comprobar una unión territorial y calcular un rendimiento con el denominador adecuado.

| Símbolo | Uso |
|:---:|---|
| 📘 | Concepto o función de apoyo. |
| ✅ | Evidencia que debe quedar visible. |
| ⚠️ | Límite de interpretación. |

Esta plantilla se encuentra en `ejercicios/PD03_integracion_territorial.ipynb`. Antes de comenzar, se crea una copia en `soluciones/PD03_integracion_territorial.ipynb` dentro del repositorio de cada estudiante. El desarrollo y la entrega se realizan en esa copia; la plantilla de `ejercicios/` se conserva como enunciado. En JupyterLab se puede duplicar el archivo y mover la copia a `soluciones/`, manteniendo el nombre indicado; si ya existe una solución, se continúa sobre ella. Las instrucciones se presentan en celdas Markdown; las celdas de código marcadas como **Desarrollo** contienen el espacio de trabajo. Cada actividad incluye una celda para explicar el resultado.

Se utiliza el kernel del entorno `.venv` del curso. Los CSV ya deben estar descargados mediante `kit/00_datos.py`. La celda inicial prepara las rutas y carga únicamente las entradas de este ejercicio; el desarrollo del análisis corresponde al estudiante. Este notebook se ejecuta de forma independiente.

**Apoyo en el manual:** secciones 4, 7, 9, 10 y la receta de razón agregada de la sección 11.

## Ubicación de trabajo: ruta visible

La carpeta base es `/mnt/c/Users/TUPTC/bigdata` y el repositorio está dentro de `big-data-postgraduate`. Desde Ubuntu:

```bash
cd /mnt/c/Users/TUPTC/bigdata
cd big-data-postgraduate
source .venv/bin/activate
jupyter lab
```

La siguiente celda declara `REPO` como una cadena de texto que puedes cambiar. `RAW` contiene la carpeta de los CSV y `OUT` la carpeta de tus resultados. No se busca el repositorio automáticamente ni se modifica el directorio del kernel. Si aparece `FileNotFoundError`, revisa la ruta impresa y el nombre del archivo. Las rutas absolutas funcionan aunque el notebook esté en `ejercicios/` o `soluciones/`.

## Preparación

Lee primero las rutas y ajusta `REPO` a tu copia. Después ejecuta cada lectura por separado: `sep` define el separador, `dtype=str` preserva códigos y `keep_default_na=False` conserva vacíos y marcadores. Observa el DataFrame antes de desarrollar las actividades. Cada llamada a `read_csv` muestra sus opciones completas.

In [1]:
import pandas as pd
import os

REPO = "/mnt/c/Users/TUPTC/bigdata/big-data-postgraduate"
RAW = REPO + "/kit/data/raw"
OUT = REPO + "/kit/salidas/soluciones/PD03"
print("Datos:", RAW)
print("Salidas:", OUT)
print("pandas:", pd.__version__)
# Crear solamente la carpeta destinada a tus resultados.
os.makedirs(OUT, exist_ok=True)

Datos: /mnt/c/Users/TUPTC/bigdata/big-data-postgraduate/kit/data/raw
Salidas: /mnt/c/Users/TUPTC/bigdata/big-data-postgraduate/kit/salidas/soluciones/PD03
pandas: 2.2.3


In [2]:
eva = pd.read_csv(
    RAW + "/eva.csv",
    sep=",",
    encoding="utf-8-sig",
    dtype=str,
    keep_default_na=False
)
display(eva.head())
print(eva.shape)

,Código Dane departamento,Departamento,Código Dane municipio,Municipio,Grupo cultivo,Subgrupo,Cultivo,Desagregación cultivo,Año,Periodo,Área sembrada,Área cosechada,Producción,Rendimiento,Ciclo del cultivo,Estado físico del cultivo,Código del cultivo,Nombre científico del cultivo
0,05,Antioquia,05001,Medellín,Hortalizas,Hortalizas de tallo,Apio,Apio,2021,2021A,8.00,8.00,128.00,16.00,Transitorio,En fresco,1050600,Apium graveolens
1,05,Antioquia,05001,Medellín,Hortalizas,Hortalizas de tallo,Apio,Apio,2021,2021B,8.00,8.00,152.00,19.00,Transitorio,En fresco,1050600,Apium graveolens
2,05,Antioquia,05001,Medellín,Hortalizas,Hortalizas de tallo,Apio,Apio,2022,2022A,8.00,8.00,127.44,15.93,Transitorio,En fresco,1050600,Apium graveolens
3,05,Antioquia,05001,Medellín,Hortalizas,Hortalizas de tallo,Apio,Apio,2022,2022B,8.00,8.00,152.00,19.00,Transitorio,En fresco,1050600,Apium graveolens
4,05,Antioquia,05001,Medellín,Hortalizas,Hortalizas de tallo,Apio,Apio,2023,2023A,8.00,7.00,133.00,19.00,Transitorio,En fresco,1050600,Apium graveolens


(166732, 18)


In [3]:
divipola = pd.read_csv(
    RAW + "/divipola.csv",
    sep=",",
    encoding="utf-8-sig",
    dtype=str,
    keep_default_na=False
)
display(divipola.head())
print(divipola.shape)

,Código Departamento,Nombre Departamento,Código Municipio,Nombre Municipio,Tipo: Municipio / Isla / Área no municipalizada,longitud,Latitud
0,05,ANTIOQUIA,05001,MEDELLÍN,Municipio,"-75,581775","6,246631"
1,05,ANTIOQUIA,05002,ABEJORRAL,Municipio,"-75,428739","5,789315"
2,05,ANTIOQUIA,05004,ABRIAQUÍ,Municipio,"-76,064304","6,632282"
3,05,ANTIOQUIA,05021,ALEJANDRÍA,Municipio,"-75,141346","6,376061"
4,05,ANTIOQUIA,05030,AMAGÁ,Municipio,"-75,702188","6,038708"


(1122, 7)


## 1. Dimensión territorial · 10 minutos

Crear `codigo_municipio` desde `Código Municipio` de DIVIPOLA. Comprobar cinco cifras y unicidad. Contar los valores de `Tipo: Municipio / Isla / Área no municipalizada`. Construir `dimension` con código, nombre departamental, nombre municipal y clasificación territorial.

📘 **Apoyo:** `.str.strip`, `.str.fullmatch(r"[0-9]{5}")`, `is_unique` y `value_counts`.

✅ **Comprobación:** La dimensión conserva las 1.122 entidades y sus tres tipos. La columna de código es única y no contiene vacíos.

In [6]:
# Desarrollo de la actividad 1
# Preparar dimension y mostrar la distribución por tipo.
divipola['codigo_municipio'] = divipola['Código Municipio'].astype(str).str.strip()
formato_valido = divipola['codigo_municipio'].str.fullmatch(r"[0-9]{5}")
assert formato_valido.all(), "¡Error! Existen códigos de municipio que no cumplen con el formato de 5 dígitos."
assert divipola['codigo_municipio'].is_unique, "¡Error! La columna 'codigo_municipio' contiene valores duplicados."
col_tipo = [c for c in divipola.columns if 'tipo' in c.lower() or 'clasifi' in c.lower()][0]
print(f"Columna de tipo detectada automáticamente: '{col_tipo}'")
print("\nDistribución por tipo territorial:")
distribucion_tipo = divipola[col_tipo].value_counts(dropna=False)
display(distribucion_tipo)
dimension = divipola[['codigo_municipio', 'Nombre Departamento', 'Nombre Municipio', col_tipo]].copy()
dimension = dimension.rename(columns={col_tipo: 'Tipo'}) 

# Comprobaciones finales requeridas
assert len(dimension) == 1122, f"Se esperaban 1.122 entidades, pero se encontraron {len(dimension)}."
assert not dimension['codigo_municipio'].isna().any() and not (dimension['codigo_municipio'] == '').any(), "La columna de código contiene valores vacíos."
print(f"\n[Comprobación Exitosa] La dimensión conserva exactamente {len(dimension)} entidades. El código es único y no contiene vacíos.")
display(dimension.head())



Columna de tipo detectada automáticamente: 'Tipo: Municipio / Isla / Área no municipalizada'

Distribución por tipo territorial:


Tipo: Municipio / Isla / Área no municipalizada
Municipio                 1103
Área no municipalizada      18
Isla                         1
Name: count, dtype: int64


[Comprobación Exitosa] La dimensión conserva exactamente 1122 entidades. El código es único y no contiene vacíos.


,codigo_municipio,Nombre Departamento,Nombre Municipio,Tipo
0,05001,ANTIOQUIA,MEDELLÍN,Municipio
1,05002,ANTIOQUIA,ABEJORRAL,Municipio
2,05004,ANTIOQUIA,ABRIAQUÍ,Municipio
3,05021,ANTIOQUIA,ALEJANDRÍA,Municipio
4,05030,ANTIOQUIA,AMAGÁ,Municipio


**Interpretación del resultado:**

se genera una columa para el codigo del municipio la cual permite garantizar que haya un solo registro y quitar los registros vacios, se realiza el conteo para auditar la frecuencia de las categorias que estan presentes en el archivo base, se evidencia el total de datos con un total de 1122 datos.

## 2. Códigos EVA sin correspondencia · 15 minutos

Preparar `codigo_municipio` desde `Código Dane municipio` y `anio` desde `Año` de EVA. Conservar originales; los códigos vacíos o mal formados deben quedar identificados para revisión. Completar ceros solo en textos numéricos válidos de menor longitud. Crear combinaciones distintas de código y año y realizar un anti-join frente a dimension. Guardar el resultado en `sin_correspondencia`.

📘 **Apoyo:** `.str.fullmatch`, `.where`, `.str.zfill`, `drop_duplicates` y `merge(how="left", indicator=True, validate="many_to_one")`.

✅ **Comprobación:** El código municipal es la clave de unión. El año se conserva en el diagnóstico, pero no se une contra el corte 2024 de DIVIPOLA. El listado se muestra aunque quede vacío.

In [8]:
# Desarrollo de la actividad 2
# Preparar código y año en eva
codigo_raw = eva['Código Dane municipio'].astype(str).str.strip()
es_numerico = codigo_raw.str.fullmatch(r"[0-9]{1,5}")
eva['codigo_municipio'] = codigo_raw.where(~es_numerico, codigo_raw.str.zfill(5))
anio_limpio = eva['Año'].astype(str).str.strip().str.replace(',', '.', regex=False)
eva['anio'] = pd.to_numeric(anio_limpio, errors='coerce').astype('Int64')
eva_combinaciones = eva[['codigo_municipio', 'anio']].drop_duplicates()
merged = eva_combinaciones.merge(
    dimension[['codigo_municipio']], 
    on='codigo_municipio', 
    how='left', 
    indicator=True, 
    validate='many_to_one'
)

# Obtener sin_correspondencia con código municipal y año.
sin_correspondencia = merged[merged['_merge'] == 'left_only'].copy()

print(f"Total de combinaciones (código de municipio y año) sin correspondencia en DIVIPOLA: {len(sin_correspondencia)}")
display(sin_correspondencia.head(10))


Total de combinaciones (código de municipio y año) sin correspondencia en DIVIPOLA: 0


,codigo_municipio,anio,_merge


**Interpretación del resultado:**

Se realiza la limpieza de los campos numericos cortos mediante el comando .str.zfill teniendo en cuenta los valores atipicos o vacios para dejar en su revision, se realizan combinaciones temporales y espaciales a traves de drop, se ailan las duplas teniendo en cuenta el codigo del municipio, el año y el reporte agropecuario, por medio del cruce left con el indicador TRUE se aislan las combinanciones de la base de datos EVA que no homologan el marco oficial el cual permite auditar los datos sin altear la informacion.

## 3. Selección para el indicador · 10 minutos

Convertir `Área cosechada` y `Producción` a `area_ha` y `produccion_t`. Seleccionar primero `Cultivo == "Papa"` y año 2024 en `papa_2024`. De ese conjunto, conservar en `apta` las filas con área positiva y producción no negativa. Contar las filas excluidas del indicador dentro de papa_2024.

📘 **Apoyo:** `pd.to_numeric`, `.eq`, `.gt`, `.ge`, `.loc` y `.copy()`.

✅ **Comprobación:** El total de papa_2024 es igual a las filas aptas más las excluidas del indicador. La ausencia de una medida no se reemplaza por cero.

In [15]:
# Desarrollo de la actividad 3
# Crear papa_2024 y apta.
anio_limpio = eva['Año'].astype(str).str.strip().str.replace(',', '.', regex=False)
eva['anio'] = pd.to_numeric(anio_limpio, errors='coerce').astype('Int64')
area_limpio = eva['Área cosechada'].astype(str).str.strip().str.replace(',', '.', regex=False)
eva['area_ha'] = pd.to_numeric(area_limpio, errors='coerce')
prod_limpio = eva['Producción'].astype(str).str.strip().str.replace(',', '.', regex=False)
eva['produccion_t'] = pd.to_numeric(prod_limpio, errors='coerce')
mascara_papa_2024 = eva['Cultivo'].eq("Papa") & eva['anio'].eq(2024)
papa_2024 = eva.loc[mascara_papa_2024].copy()
mascara_apta = papa_2024['area_ha'].gt(0) & papa_2024['produccion_t'].ge(0)
apta = papa_2024.loc[mascara_apta].copy()
total_papa_2024 = len(papa_2024)
total_apta = len(apta)
total_excluidas = total_papa_2024 - total_apta
print(f"Total de registros en papa_2024: {total_papa_2024}")
print(f"Total de registros aptos (válidos para el indicador): {total_apta}")
print(f"Total de filas excluidas del indicador: {total_excluidas}")

# Mostrar las exclusiones del indicador y conciliar los conteos.
assert total_papa_2024 == total_apta + total_excluidas, "¡Error! La conciliación de conteos no coincide."
print("\n[Comprobación Exitosa] El total de papa_2024 es igual a la suma de filas aptas más las excluidas.")


Total de registros en papa_2024: 831
Total de registros aptos (válidos para el indicador): 779
Total de filas excluidas del indicador: 52

[Comprobación Exitosa] El total de papa_2024 es igual a la suma de filas aptas más las excluidas.


**Interpretación del resultado:**



Filtrado temporal y sectorial: Se logró aislar el universo correspondiente al cultivo de papa para el año 2024 (papa_2024) mediante condiciones lógicas exactas.

Depuración para indicadores de rendimiento (apta): Se restringió el subconjunto a aquellas observaciones con coherencia agronómica (área cosechada estrictamente mayor a cero y producción mayor o igual a cero), excluyendo de forma controlada los registros con valores nulos o inconsistencias numéricas sin descartar la trazabilidad general.

Conciliación numérica: La verificación confirmó que el tamaño total del grupo base (papa_2024) equivale de forma perfecta a la suma de los registros aptos y las exclusiones identificadas.

## 4. Rendimiento agregado · 15 minutos

Agrupar apta por código municipal, cultivo, estado físico del cultivo y año. Crear `rendimiento` con cantidad de registros, suma de producción y suma de área cosechada. Calcular `rendimiento_t_ha` dividiendo ambas sumas.

📘 **Apoyo:** `groupby(..., as_index=False, dropna=False)`, `agg`, `size` y `sum`.

✅ **Comprobación:** Cada grupo tiene área total positiva. La suma de los conteos por grupo coincide con len(apta). El indicador se calcula como suma de producción / suma de área, conservando cultivo y estado físico en la agrupación.

In [16]:
# Desarrollo de la actividad 4
# Crear rendimiento con los agregados y rendimiento_t_ha.

# --- Desarrollo de la actividad 4 ---

# 1. Agrupar 'apta' por código municipal, cultivo, estado físico y año, conservando grupos con dropna=False
rendimiento = apta.groupby(
    ['codigo_municipio', 'Cultivo', 'Estado físico del cultivo', 'anio'],
    as_index=False,
    dropna=False
).agg(
    registros=('produccion_t', 'size'),
    produccion_t=('produccion_t', 'sum'),
    area_ha=('area_ha', 'sum')
)

# 2. Calcular rendimiento_t_ha dividiendo la suma de producción entre la suma de área cosechada
rendimiento['rendimiento_t_ha'] = rendimiento['produccion_t'] / rendimiento['area_ha']

# 3. Comprobaciones requeridas
# Verificar que cada grupo tiene un área total estrictamente positiva
assert (rendimiento['area_ha'] > 0).all(), "¡Error! Se encontraron grupos con área total menor o igual a cero."

# Verificar que la suma de los conteos por grupo coincide exactamente con len(apta)
assert rendimiento['registros'].sum() == len(apta), f"¡Error! La suma de registros ({rendimiento['registros'].sum()}) no coincide con len(apta) ({len(apta)})."

print(f"[Comprobación Exitosa] Rendimiento agregado calculado correctamente.")
print(f"Total de grupos consolidados: {len(rendimiento)}")
display(rendimiento.head(10))


[Comprobación Exitosa] Rendimiento agregado calculado correctamente.
Total de grupos consolidados: 292


,codigo_municipio,Cultivo,Estado físico del cultivo,anio,registros,produccion_t,area_ha,rendimiento_t_ha
0,05001,Papa,En fresco,2024,2,1280.0,80.0,16.000000
1,05002,Papa,En fresco,2024,2,1650.0,165.0,10.000000
2,05004,Papa,En fresco,2024,4,15.0,5.0,3.000000
3,05079,Papa,En fresco,2024,2,268.0,34.0,7.882353
4,05086,Papa,En fresco,2024,3,10550.0,905.0,11.657459
5,05088,Papa,En fresco,2024,2,1710.0,95.0,18.000000
6,05093,Papa,En fresco,2024,1,6.0,2.0,3.000000
7,05148,Papa,En fresco,2024,4,14757.0,789.0,18.703422
8,05206,Papa,En fresco,2024,2,1200.0,80.0,15.000000
9,05237,Papa,En fresco,2024,3,2638.0,103.8,25.414258


**Interpretación del resultado:**



Agregación multidimensional: Se consolidó la información a nivel territorial y agronómico utilizando groupby con as_index=False y dropna=False, asegurando que no se excluyera ninguna categoría o valor ausente de la estructura agrupada.

Métricas de producción y superficie: Mediante las funciones de agregación se obtuvieron los totales de producción y área cosechada por grupo, permitiendo derivar el indicador de eficiencia productiva (rendimiento_t_ha).

Validación de integridad: Las aserciones (assert) comprobaron exitosamente que todos los grupos formados cuentan con un área total estrictamente mayor a cero y que la suma de los registros individuales concuerda de forma exacta con el tamaño del conjunto base apta.

## 5. Unión con nombres y clasificación · 10 minutos

Relacionar rendimiento con dimension mediante un left merge por codigo_municipio. Declarar `validate="many_to_one"` e incorporar un indicador de correspondencia. Guardar el resultado en `rendimiento_municipal` y mostrar los códigos sin nombre territorial, si existen.

📘 **Apoyo:** `merge`, `validate`, `indicator` y `len`.

✅ **Comprobación:** La cantidad de filas antes y después de la unión debe ser idéntica. Un fallo de cardinalidad debe investigarse; no se resuelve eliminando arbitrariamente claves de la dimensión.

In [17]:
# Desarrollo de la actividad 5
# Unir rendimiento con dimension.
# --- Desarrollo de la actividad 5 ---

# 1. Unir rendimiento con dimension mediante un left merge por codigo_municipio
rendimiento_municipal = rendimiento.merge(
    dimension,
    on='codigo_municipio',
    how='left',
    validate='many_to_one',
    indicator=True
)

# 2. Verificar la conservación estricta del número de filas
assert len(rendimiento) == len(rendimiento_municipal), (
    f"¡Error de cardinalidad! Filas antes: {len(rendimiento)}, filas después: {len(rendimiento_municipal)}"
)

# 3. Identificar si existen códigos sin nombre territorial (registros exclusivos de la izquierda)
sin_nombre = rendimiento_municipal[rendimiento_municipal['_merge'] == 'left_only']

print(f"Total de filas antes de la unión: {len(rendimiento)}")
print(f"Total de filas después de la unión: {len(rendimiento_municipal)}")
print(f"Registros sin correspondencia en la dimensión: {len(sin_nombre)}")

if len(sin_nombre) > 0:
    print("Se encontraron códigos sin nombre territorial asociado:")
    display(sin_nombre[['codigo_municipio']].drop_duplicates())
else:
    print("\n[Comprobación Exitosa] Cobertura territorial completa: todos los registros tienen correspondencia exacta en 'dimension'.")

# Mostrar una muestra del resultado final integrado
display(rendimiento_municipal.head(10))
# Verificar la conservación de filas y presentar la cobertura de la unión.

Total de filas antes de la unión: 292
Total de filas después de la unión: 292
Registros sin correspondencia en la dimensión: 0

[Comprobación Exitosa] Cobertura territorial completa: todos los registros tienen correspondencia exacta en 'dimension'.


,codigo_municipio,Cultivo,Estado físico del cultivo,anio,registros,produccion_t,area_ha,rendimiento_t_ha,Nombre Departamento,Nombre Municipio,Tipo,_merge
0,05001,Papa,En fresco,2024,2,1280.0,80.0,16.000000,ANTIOQUIA,MEDELLÍN,Municipio,both
1,05002,Papa,En fresco,2024,2,1650.0,165.0,10.000000,ANTIOQUIA,ABEJORRAL,Municipio,both
2,05004,Papa,En fresco,2024,4,15.0,5.0,3.000000,ANTIOQUIA,ABRIAQUÍ,Municipio,both
3,05079,Papa,En fresco,2024,2,268.0,34.0,7.882353,ANTIOQUIA,BARBOSA,Municipio,both
4,05086,Papa,En fresco,2024,3,10550.0,905.0,11.657459,ANTIOQUIA,BELMIRA,Municipio,both
5,05088,Papa,En fresco,2024,2,1710.0,95.0,18.000000,ANTIOQUIA,BELLO,Municipio,both
6,05093,Papa,En fresco,2024,1,6.0,2.0,3.000000,ANTIOQUIA,BETULIA,Municipio,both
7,05148,Papa,En fresco,2024,4,14757.0,789.0,18.703422,ANTIOQUIA,EL CARMEN DE VIBORAL,Municipio,both
8,05206,Papa,En fresco,2024,2,1200.0,80.0,15.000000,ANTIOQUIA,CONCEPCIÓN,Municipio,both
9,05237,Papa,En fresco,2024,3,2638.0,103.8,25.414258,ANTIOQUIA,DONMATÍAS,Municipio,both


**Interpretación del resultado:**



Enlace geoespacial y productivo: Se integró el consolidado de rendimiento departamental con el marco territorial oficial (dimension) a través del identificador único codigo_municipio, preservando todas las variables descriptivas (departamento, municipio y tipo).

Control de integridad y cardinalidad: La validación estricta (validate="many_to_one") en conjunto con la aserción de longitudes (len) confirmó que la estructura relacional operó correctamente sin duplicar ni descartar filas de forma indebida.

Auditoría de cobertura: El análisis del indicador de fusión permitió comprobar si existen anomalías o vacíos de correspondencia entre los reportes agropecuarios y el catálogo geográfico oficial.

## 6. Exportación e interpretación · 10 minutos

Exportar rendimiento_municipal a `OUT + "/rendimiento_municipal.csv"` y sin_correspondencia a `OUT + "/codigos_sin_correspondencia.csv"`, ambos sin índice. Explicar por qué se dividen sumas en lugar de promediar rendimientos individuales y qué limitación introduce comparar EVA con un catálogo territorial de corte 2024.

📘 **Apoyo:** `to_csv` y los resultados de las actividades anteriores.

✅ **Comprobación:** El archivo de códigos conserva sus cabeceras aunque no tenga filas. La explicación distingue una coincidencia de códigos de una equivalencia histórica de límites.

In [18]:
# Desarrollo de la actividad 6
# Exportar las dos tablas.
# --- Desarrollo de la actividad 6 ---
import os

# 1. Asegurar la existencia del directorio de salida
os.makedirs(OUT, exist_ok=True)

# 2. Exportar rendimiento_municipal sin índice
ruta_rendimiento = os.path.join(OUT, "rendimiento_municipal.csv")
rendimiento_municipal.to_csv(ruta_rendimiento, index=False)

# 3. Exportar sin_correspondencia sin índice (conserva las cabeceras incluso si el DataFrame está vacío)
ruta_sin_corr = os.path.join(OUT, "codigos_sin_correspondencia.csv")
sin_correspondencia.to_csv(ruta_sin_corr, index=False)

print("[Comprobación Exitosa] Archivos exportados correctamente en la ruta de salidas:")
print(f" - {ruta_rendimiento}")
print(f" - {ruta_sin_corr}")
# Explicar el denominador y la limitación temporal en la celda siguiente.

[Comprobación Exitosa] Archivos exportados correctamente en la ruta de salidas:
 - /mnt/c/Users/TUPTC/bigdata/big-data-postgraduate/kit/salidas/soluciones/PD03/rendimiento_municipal.csv
 - /mnt/c/Users/TUPTC/bigdata/big-data-postgraduate/kit/salidas/soluciones/PD03/codigos_sin_correspondencia.csv


**Interpretación del resultado:**




Calcular el rendimiento agregado como la división entre la suma total de la producción y la suma total del área cosechada ($\frac{\sum \text{producción}}{\sum \text{área}}$) garantiza la obtención de un promedio ponderado real por el tamaño de la superficie. Promediar rendimientos individuales ($\text{promedio}(\frac{\text{prod}_i}{\text{área}_i})$ introduciría un sesgo estadístico severo (paradoja o distorsión por ponderación), donde pequeñas parcelas con rendimientos atípicos (muy altos o muy bajos) distorsionarían la verdadera eficiencia productiva global del grupo analizado.¿Qué limitación introduce comparar EVA con un catálogo territorial de corte 2024?Comparar registros históricos o multi-anuales de EVA contra un catálogo DIVIPOLA estático de corte 2024 confunde discrepancias de codificación con cambios histórico-administrativos en los límites territoriales. A lo largo del tiempo, los municipios pueden segregarse, fusionarse o cambiar de código Dane oficial. Por lo tanto, un código ausente en el corte 2024 no siempre representa un error de captura en los datos agropecuarios, sino una obsolescencia temporal del identificador municipal debido a modificaciones en la división político-administrativa del país.

## Entrega

El notebook completado se conserva como `soluciones/PD03_integracion_territorial.ipynb`. Las salidas regenerables se guardan en `kit/salidas/soluciones/PD03/`.

✅ Deben quedar visibles el control de la dimensión, el anti-join, el conteo de exclusiones, la conciliación de grupos y la conservación de filas en el merge.

⚠️ Una coincidencia de códigos no prueba estabilidad de límites municipales a través del tiempo. El rendimiento agregado se interpreta para el cultivo y estado físico seleccionados; no debe sumarse con rendimientos de otros cultivos.